# Notebook 1 - baseline-ul corectat

## Ce raspunde

Doua lucruri din lista recenzentului, cu acelasi set de rulari:

**"Refa mai intai baseline-ul dupa corectarea evaluarii si a pragului. Acesta
trebuie sa ramana punctul principal de referinta."** Rularea de aici e prima cu
toate cele sapte corectii active simultan.

Trei rulari, in ordinea in care conteaza daca sesiunea se termina devreme:

| rulare | pasi | ce da |
|---|---|---|
| seed 42 | 49 390 | numarul de referinta corectat |
| seed 43 | 49 390 | a doua tragere, deci podeaua de zgomot |
| calibrare, seed 42 | 98 780 | daca 49 390 e lungimea potrivita |

Doua seed-uri dau o abatere standard cu un singur grad de libertate, ceea ce e
putin - dar se poate combina cu cele trei seed-uri comise, care sunt un al
doilea esantion independent al aceleiasi marimi. Celula de analiza tipareste
raportul variantelor inainte de a combina si refuza combinarea daca cele doua
protocoale au imprastieri prea diferite.

Si, gratis, la final: greutatile finale recoapte scorate langa cele alese pe
validare, ca sa se vada cat din cifra raportata e noroc de selectie.

## Ce e corectat fata de baseline-ul comis

| | comis | aici |
|---|---|---|
| prag ales in | spatiul 192 x 192 | geometria originala |
| felii fara predictie | umplute cu zero | refuzate |
| selectia checkpointului | validare `all` | validare `all` |
| buget | 100 epoci, oprire dupa 20 de epoci | 49 390 de pasi, oprire dupa 9 000 de pasi |
| cosinus | peste 50 de **epoci** | peste 49 390 de **pasi** |
| verificare non-finit | dupa `backward()` | inainte |

## De ce conteaza numarul de la final

Cele trei seed-uri comise ale baseline-ului dau 0.5201, 0.4898 si 0.4901 -
abatere standard 0.0174, deci o banda de aproximativ +/-0.048 pe o diferenta
intre doua rulari. Grila de esantionare din notebook-ul W se intinde, toata, pe
0.074, iar `all` bate `1:3` cu 0.026.

Trei rulari dau o abatere standard cu doua grade de libertate, ceea ce abia se
cheama estimare: propriul ei interval de incredere se intinde pe un factor de
vreo doisprezece. Cele trei rulari comise sunt insa un al doilea esantion
independent al aceleiasi marimi - difera intre ele doar prin seed, verificat
cheie cu cheie in `config.json`. Combinate, dau patru grade de libertate, cam
cat ar fi dat cinci rulari noi, la costul a trei. Celula de analiza tipareste si
raportul variantelor, ca sa se vada daca combinarea e legitima in loc sa fie
presupusa.

In [ ]:
!pip install -q monai==1.6.0 nibabel==5.4.2

In [ ]:
import os, sys, time, torch

SESSION_START = time.time()

def find_code_root():
    """The uploaded repository: the directory holding src/config.py."""
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "src" and "config.py" in files:
            return os.path.dirname(root)
    raise FileNotFoundError("Nu gasesc src/config.py sub /kaggle/input. "
                            "Ataseaza dataset-ul cu codul.")

def find_raw_root():
    """
    The raw Decathlon archive: dataset.json sitting next to imagesTr/.

    The code dataset also ships an archive/dataset.json as a layout reference,
    so matching on that filename alone would find the wrong directory. Requiring
    imagesTr/ as a sibling separates them.
    """
    for root, dirs, files in os.walk("/kaggle/input"):
        if "dataset.json" in files and "imagesTr" in dirs:
            return root
    raise FileNotFoundError(
        "Nu gasesc arhiva bruta (dataset.json langa imagesTr/). Ataseaza "
        "https://www.kaggle.com/datasets/vivekprajapati2048/medical-segmentation-decathlon-lung")

CODE_ROOT = find_code_root()
RAW_ROOT = find_raw_root()
sys.path.insert(0, CODE_ROOT)

assert torch.cuda.is_available(), "Nicio placa video. Settings -> Accelerator -> GPU T4 x2."
cap = torch.cuda.get_device_capability(0)
assert cap >= (7, 0), (
    f"{torch.cuda.get_device_name(0)} are compute capability sm_{cap[0]}{cap[1]}, "
    "iar PyTorch cere sm_70+. Settings -> Accelerator -> GPU T4 x2, apoi reporneste.")

print("GPU: ", torch.cuda.get_device_name(0), f"(sm_{cap[0]}{cap[1]})")
print("Code:", CODE_ROOT)
print("Raw: ", RAW_ROOT)

In [ ]:
import shutil, json

# set_data_dir must run before any other src module is imported: they bind
# OUTPUT_DIR and DATA_DIR by value at import time, so repointing config after
# that would leave them looking at the wrong place.
import src.config as config
config.set_data_dir(RAW_ROOT)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

for fname in ("patient_split.json", "eda_statistics.csv"):
    src_path = os.path.join(CODE_ROOT, "output", fname)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(config.OUTPUT_DIR, fname))

split = json.load(open(os.path.join(config.OUTPUT_DIR, "patient_split.json")))
print(f"Split: {len(split['train'])} train / {len(split['val'])} val / {len(split['test'])} test")
assert len(split["train"]) == 44 and len(split["val"]) == 9 and len(split["test"]) == 10, \
    "Split diferit de cel al experimentelor existente - comparatia ar fi confundata."

### Verificare versiune de cod
Sapte verificari. Fara ele notebook-ul ar rula tacut cu vechiul scheduler pe epoci si ar raporta ca a rulat cu cel pe pasi.

In [ ]:
import inspect
import numpy as np
from src.training.train import run_training_experiment, EarlyStopping
from src.training.dataset import LungSliceDataset
from src.evaluation.metrics import (threshold_sweep_original_geometry,
                                    stack_slice_predictions,
                                    compute_all_3d_metrics)

sig = inspect.signature(run_training_experiment).parameters
for needed in ("max_steps", "negative_ratio", "preprocessed_name",
               "metadata_name", "schedule_unit", "patience_steps", "min_steps"):
    assert needed in sig, f"run_training_experiment nu accepta {needed} - zip vechi."

# The scheduler has to move between batches, not only between epochs. A
# parameter that is accepted and ignored would leave the epoch-unit schedule in
# place and every "equal steps" claim below would be false.
assert "scheduler" in inspect.signature(
    __import__("src.training.train", fromlist=["x"]).train_one_epoch).parameters
stopper = EarlyStopping(patience=2, min_epochs=0, patience_steps=100, min_steps=0)
stopper.step(0.5, epoch=1, steps_taken=10)
assert not stopper.step(0.4, epoch=2, steps_taken=20), \
    "patience-ul in pasi nu ajunge la stopper - se opreste dupa epoci"

# The sweep must run in the geometry the result is reported in.
assert "require_full_coverage" in inspect.signature(
    threshold_sweep_original_geometry).parameters

# A slice the model never saw must not be scored as confidently empty.
try:
    stack_slice_predictions({0: np.zeros((4, 4), np.float32)}, n_slices=3)
    raise AssertionError("garda de acoperire lipseste - zip vechi")
except ValueError:
    pass

# Raportul volum prezis / volum real, cerut in raportare.
gt = np.zeros((8, 8, 4), np.uint8); gt[2:5, 2:5, 1] = 1
pred = np.zeros((8, 8, 4), np.uint8); pred[2:4, 2:5, 1] = 1
m = compute_all_3d_metrics(pred, gt, surface_metrics=False)
assert "volume_ratio_3d" in m and abs(m["volume_ratio_3d"] - 6 / 9) < 1e-9

print("Cod OK - inclusiv scheduler pe pas, patience pe pas, raport de volum.")

In [ ]:
import subprocess
r = subprocess.run(["python", "-m", "pytest", "tests", "-q"], cwd=CODE_ROOT,
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, "Testele pica - nu continua."

## Date
Dataset-ul preprocesat de 192 x 192, atasat. Nimic nu se reconstruieste.

In [ ]:
def find_preprocessed_root():
    """
    An attached preprocessed dataset: the directory holding preprocessed/index.json
    with a metadata/ sibling.
    """
    for root, dirs, files in os.walk("/kaggle/input"):
        if (os.path.exists(os.path.join(root, "preprocessed", "index.json"))
                and os.path.isdir(os.path.join(root, "metadata"))):
            return root
    return None

PRE_ROOT = find_preprocessed_root()
assert PRE_ROOT, ("Ataseaza dataset-ul preprocesat de 192. Partea 1 il foloseste "
                  "direct si nu are de ce sa il reconstruiasca.")

for name in ("preprocessed", "metadata"):
    dst = os.path.join(config.OUTPUT_DIR, name)
    if os.path.lexists(dst):
        (shutil.rmtree if os.path.isdir(dst) and not os.path.islink(dst)
         else os.unlink)(dst)
    os.symlink(os.path.join(PRE_ROOT, name), dst)

index = json.load(open(os.path.join(config.OUTPUT_DIR, "preprocessed", "index.json")))
total = sum(index["cases"][c]["n_slices"] for c in index["cases"])
pos = sum(len(index["cases"][c]["positive_slices"]) for c in index["cases"])
assert (total, pos) == (20573, 1920), f"dataset diferit: {total}, {pos}"
print(f"  {len(index['cases'])} pacienti | {total} felii | {pos} pozitive")

## Bugetul si regula de oprire, in pasi

In [ ]:
BASELINE_STEPS = 49_390     # 55 epoci x 898 la esantionare `all`, bugetul de referinta
BATCH = 16

# Oprirea timpurie e OPRITA, si asta e o corectie fata de prima incercare.
#
# Prima varianta a acestui notebook a folosit 9 000 de pasi de rabdare. Cele
# trei seed-uri au cheltuit 53%, 65% si 76% din acelasi buget si si-au pastrat
# checkpoint-ul la rate de invatare de 7.6e-4, 5.8e-4 si 4.1e-4 - in timp ce
# baseline-ul comis, cu care se compara, si-l pastrase la 2.14e-4 dupa ce
# cheltuise bugetul intreg. Rezultatul: 0.4388 in loc de 0.5000, cu o
# imprastiere de trei ori mai mare.
#
# Un experiment al carui control este numarul de pasi nu se poate opri in alt
# punct la fiecare rulare. Atunci rularile nu mai sunt egalizate pe pasi, iar
# unde se opresc devine ea insasi o sursa de variatie. Bugetul se cheltuie
# integral; alegerea epocii ramane treaba checkpoint-ului, nu a opririi.
#
# Garda de divergenta si detectorul de colaps raman active - acelea sunt masuri
# de siguranta impotriva unei rulari moarte, nu reguli de oprire.
PATIENCE_STEPS = 0        # 0 = fara oprire timpurie
MIN_STEPS = 0

COMMON = dict(
    model_type="unet", loss_type="dice_ce", augment="anatomic",
    max_steps=BASELINE_STEPS, batch_size=BATCH, lr=1e-3, max_grad_norm=1.0,
    schedule_unit="step", patience_steps=PATIENCE_STEPS, min_steps=MIN_STEPS,
    postproc_min_fraction=0.10, surface_metrics=False, num_workers=2,
)
print(f"  Buget      : {BASELINE_STEPS:,} pasi de optimizare")
print(f"  Scheduler  : cosinus peste pasi, nu peste epoci")
print(f"  Oprire     : "
      + ("dezactivata - bugetul se cheltuie integral" if not PATIENCE_STEPS
         else f"{PATIENCE_STEPS:,} pasi rabdare, prag {MIN_STEPS:,} pasi"))

In [ ]:
import numpy as np


def all_slice_block(rep):
    """
    The evaluation that saw every slice, wherever the report keeps it.

    `run_training_experiment` writes the primary protocol at the top level and
    the second one under "second_protocol". *Which* of the two saw every slice
    depends on `eval_sampling`: with `eval_sampling="positives"` the top-level
    block is the oracle one, zero-filled on every slice the model never ran on.

    Reading the top level unconditionally therefore compares a whole-volume
    number against an oracle number. The first run of notebook 2 did exactly
    that and printed the defective protocol as the better one - 0.5515 against
    0.3045 - when on every slice it is 0.2067 against 0.3045, the other way
    round. So the block is selected rather than assumed.

    Returns:
        tuple: (block holding per_patient_test_metrics, its threshold)
    """
    if not rep.get("oracle_positive_slices_evaluation"):
        return rep, rep["optimal_threshold"]
    second = rep.get("second_protocol")
    if second and second.get("eval_sampling") == "all":
        return second, second["optimal_threshold"]
    raise KeyError(
        "this run was evaluated on a slice subset and has no all-slice "
        "evaluation to report; re-run it with second_eval_sampling='all'")


def mean_of(block, key):
    """Macro mean over patients, skipping the NaNs a metric legitimately has."""
    per = block["per_patient_test_metrics"]
    vals = [v[key] for v in per.values() if key in v and not np.isnan(v[key])]
    return float(np.mean(vals)) if vals else float("nan")


def median_of(block, key):
    per = block["per_patient_test_metrics"]
    vals = [v[key] for v in per.values() if key in v and not np.isnan(v[key])]
    return float(np.median(vals)) if vals else float("nan")


# (label, key, format, aggregator). The volume ratio is reported as a median
# because it is a ratio over a denominator that varies by two orders of
# magnitude across patients: on one run the mean read 1.809 while eight of ten
# patients were under-segmenting, because lung_058's 188 ground-truth voxels
# produced a ratio of 9.27 on its own.
REPORT_FIELDS = [
    ("Dice 3D",            "dice_3d",             "{:.4f}", mean_of),
    ("sensibilitate",      "sensitivity_3d",      "{:.4f}", mean_of),
    ("precizie",           "precision_3d",        "{:.4f}", mean_of),
    ("vol prezis/real",    "volume_ratio_3d",     "{:.3f}", median_of),
    ("componente FP",      "fp_components",       "{:.2f}", mean_of),
    ("% felii negative cu predictie", "false_alarm_rate_2d", "{:.2%}", mean_of),
]


def report_row(name, rep):
    """The six required figures, always on the all-slice evaluation."""
    block, threshold = all_slice_block(rep)
    cells = [fmt.format(agg(block, key)) for _, key, fmt, agg in REPORT_FIELDS]
    return [name] + cells + [f"{threshold:.2f}", f"{rep['optimizer_steps']:,}"]


def print_table(rows):
    header = ["rulare"] + [n for n, _, _, _ in REPORT_FIELDS] + ["prag", "pasi"]
    widths = [max(len(str(r[i])) for r in [header] + rows)
              for i in range(len(header))]
    line = "  ".join(h.rjust(w) for h, w in zip(header, widths))
    print(line)
    print("-" * len(line))
    for row in rows:
        print("  ".join(str(c).rjust(w) for c, w in zip(row, widths)))

## Rularile

O rulare, seed 42, bugetul intreg de 49 390 de pasi, fara oprire timpurie.
Se arhiveaza imediat ce se termina.

In [ ]:
import glob, shutil, traceback, time
from src.training.train import run_training_experiment

OUT = "/kaggle/working/results_1_baseline_seeds"
os.makedirs(OUT, exist_ok=True)


def archive():
    """After every run, so a later crash cannot cost an earlier result."""
    for d in sorted(glob.glob(os.path.join(config.OUTPUT_DIR, "experiments", "*"))):
        if os.path.basename(d).startswith(("base_corrected_",
                                          "budget_calibration")):
            shutil.copytree(d, os.path.join(OUT, os.path.basename(d)),
                            dirs_exist_ok=True)
    shutil.make_archive(OUT, "zip", OUT)


# (seed, step budget, estimated minutes). Two seeds at the reference budget give
# the noise band; the third run doubles the budget to ask whether the reference
# budget is the right one. Ordered so that if the session runs short, what is
# lost is the calibration rather than the band.
LONG_STEPS = 2 * BASELINE_STEPS      # 110 epoci la 898 pasi
JOBS = [
    ("base_corrected_seed42", 42, BASELINE_STEPS,  80),
    ("base_corrected_seed43", 43, BASELINE_STEPS,  80),
    ("budget_calibration",    42, LONG_STEPS,     155),
]
SESSION_BUDGET_H = 9.5

runs, failures = {}, {}
for name, seed, steps, est_min in JOBS:
    elapsed_h = (time.time() - SESSION_START) / 3600
    if elapsed_h + est_min / 60 > SESSION_BUDGET_H:
        print(f"\n[SARIT] {name}: {elapsed_h:.1f} h scurse + ~{est_min} min "
              f"depaseste bugetul de {SESSION_BUDGET_H} h", flush=True)
        continue
    print(f"\n{'=' * 78}\n  {name}  (seed {seed}, {steps:,} pasi)  "
          f"~{est_min} min | {elapsed_h:.1f} h scurse\n{'=' * 78}", flush=True)
    t0 = time.time()
    try:
        # max_steps is overridden per job; everything else is COMMON, so the
        # long run differs from seed 42 in exactly one thing - and because the
        # cosine spans the budget, that one thing also stretches the schedule.
        # That is the point: a longer run under the same schedule is not a
        # longer schedule.
        kwargs = dict(COMMON)
        kwargs["max_steps"] = steps
        runs[name] = run_training_experiment(
            exp_name=name, seed=seed, sampling="all", eval_sampling="all",
            **kwargs)
        print(f"  {name}: {(time.time() - t0) / 60:.1f} min | "
              f"{runs[name]['optimizer_steps']:,} pasi | "
              f"{runs[name]['epochs_run']} epoci", flush=True)
    except Exception:
        failures[name] = traceback.format_exc()
        print(f"  [ESEC] {name}\n{failures[name]}", flush=True)
    archive()

# The two reference runs, keyed by seed, for the noise-band analysis below.
SEEDS = [42, 43]
reference = {seed: runs[f"base_corrected_seed{seed}"] for seed in SEEDS
             if f"base_corrected_seed{seed}" in runs}

print(f"\n{len(runs)} reusite, {len(failures)} esecuri, "
      f"{(time.time() - SESSION_START) / 3600:.1f} h scurse")

### Raportarea ceruta

Cele sase cifre pe care recenzentul le cere obligatoriu, pe evaluarea cu toate
feliile: Dice 3D, sensibilitate, precizie, raportul volum prezis / volum real,
numarul componentelor false positive, si procentul feliilor negative pe care
modelul prezice cel putin un pixel.

In [ ]:
rows = [report_row(f"seed {seed}", reference[seed])
        for seed in sorted(reference)]
print_table(rows)

### Podeaua de zgomot

In [ ]:
import statistics as st, math

# The three committed baseline seeds, rescored in original geometry. They differ
# from each other only in seed - verified key by key against config.json - so
# they are a second, independent sample of the same quantity this notebook is
# measuring, under the old protocol.
COMMITTED = [0.5201, 0.4898, 0.4901]

# Through all_slice_block, not the raw report: these runs happen to have
# eval_sampling="all" so the two coincide, but relying on that is how the
# oracle number got compared against a whole-volume one in notebook 2.
dice = {seed: mean_of(all_slice_block(rep)[0], "dice_3d")
        for seed, rep in reference.items()}
vals = list(dice.values())

print(f"  Protocol corectat, {len(vals)} seed-uri:")
print("    " + "  ".join(f"{v:.4f}" for v in sorted(vals)))
print(f"  Protocol comis,    {len(COMMITTED)} seed-uri:")
print("    " + "  ".join(f"{v:.4f}" for v in sorted(COMMITTED)))

if len(vals) > 1:
    mean_new, sd_new = st.mean(vals), st.stdev(vals)
    mean_old, sd_old = st.mean(COMMITTED), st.stdev(COMMITTED)
    print(f"\n  {'':18s} {'medie':>8s} {'ab.std':>8s} {'interval':>9s}")
    print(f"  {'corectat':18s} {mean_new:8.4f} {sd_new:8.4f} "
          f"{max(vals) - min(vals):9.4f}")
    print(f"  {'comis':18s} {mean_old:8.4f} {sd_old:8.4f} "
          f"{max(COMMITTED) - min(COMMITTED):9.4f}")

    # Three runs give an sd with two degrees of freedom, which is barely an
    # estimate: its own 95% interval spans roughly a factor of twelve. Pooling
    # the two independent samples doubles that to four, worth about as much as
    # five runs would have been - but only if the two really do have the same
    # spread, so the ratio is printed rather than assumed.
    ratio = (max(sd_new, sd_old) / min(sd_new, sd_old)) ** 2
    poolable = ratio < 10
    print(f"\n  Raportul variantelor: {ratio:.1f}x")
    print("  " + ("Comparabile - se combina." if poolable else
                  "[!] Prea diferite ca sa fie combinate. Protocolul corectat "
                  "are alta imprastiere\n      decat cel comis, si banda de mai "
                  "jos foloseste doar sirul corectat."))

    if poolable:
        df = (len(vals) - 1) + (len(COMMITTED) - 1)
        sd_band = math.sqrt(((len(vals) - 1) * sd_new ** 2
                             + (len(COMMITTED) - 1) * sd_old ** 2) / df)
        source = f"combinata din {len(vals)} + {len(COMMITTED)} rulari"
    else:
        df = len(vals) - 1
        sd_band = sd_new
        source = f"doar din cele {len(vals)} rulari corectate"
    t_df = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571}.get(df, 2.0)

    print(f"\n  Ab. standard {sd_band:.4f}  ({source}, {df} grade de libertate)")
    print(f"  Diferenta dintre doua rulari singulare: ab.std "
          f"{sd_band * math.sqrt(2):.4f}")
    print(f"  Banda 95%: +/-{t_df * sd_band * math.sqrt(2):.4f}")
    print(f"\n  Sub aceasta banda, o diferenta nu se distinge de seed.")
    print(f"  Pentru referinta, grila de esantionare din W se intinde, toata,")
    print(f"  pe 0.074 - iar `all` bate `1:3` cu 0.026 si `1:9` cu 0.037.")

    se = sd_new / math.sqrt(len(vals))
    t_new = {1: 12.706, 2: 4.303, 3: 3.182}.get(len(vals) - 1, 2.0)
    print(f"\n  Baseline-ul corectat, ca punct de referinta:")
    print(f"    {mean_new:.4f}  IC 95% [{mean_new - t_new * se:.4f}, "
          f"{mean_new + t_new * se:.4f}]")
elif not vals:
    print("\n  Nicio rulare de referinta nu a reusit - nimic de raportat aici.")
else:
    # One run gives a reference number and no variance. Saying so, and naming
    # what gets carried forward instead, beats printing a band that looks
    # measured here and was not.
    only = vals[0]
    sd_old = st.stdev(COMMITTED)
    band = 2 * sd_old * math.sqrt(2)
    print(f"\n  Baseline-ul corectat, ca punct de referinta: {only:.4f}")
    print(f"  (o singura rulare - fara interval de incredere pe ea)")
    print(f"\n  Podeaua de zgomot NU se poate masura dintr-o singura rulare.")
    print(f"  Ce se duce mai departe e banda din cele trei seed-uri comise:")
    print(f"    ab.std {sd_old:.4f}  ->  banda 95% pe o diferenta +/-{band:.4f}")
    print(f"\n  De retinut cand se citeste orice comparatie de aici incolo:")
    print(f"    - banda vine de la protocolul VECHI (cosinus pe epoci, oprire")
    print(f"      timpurie la 20 de epoci), nu de la cel folosit aici;")
    print(f"    - daca protocolul nou are alta imprastiere, banda e gresita, si")
    print(f"      nimic din ce avem nu poate spune in ce directie;")
    print(f"    - o a doua rulare pe alt seed, ~1.2 h, ar rezolva asta.")
    print(f"\n  Diferenta fata de baseline-ul comis (0.5000, media a 3 seed-uri):")
    print(f"    {only - st.mean(COMMITTED):+.4f}"
          + ("  - in banda, deci nedecis" if abs(only - st.mean(COMMITTED)) < band
             else "  - in afara benzii"))

### Pe pacient

O medie peste zece pacienti ascunde totul. Daca aceiasi doi pacienti sunt
responsabili de imprastiere la toate seed-urile, zgomotul e in evaluare, nu in
antrenare - si atunci validarea incrucisata e raspunsul, nu mai multe seed-uri.

In [ ]:
per = {seed: {c: v["dice_3d"]
              for c, v in all_slice_block(rep)[0]["per_patient_test_metrics"].items()}
       for seed, rep in reference.items()}

if not reference:
    print("  Nicio rulare de referinta.")
elif len(reference) == 1:
    # No spread to compute, but the per-patient breakdown still matters: it is
    # where lung_036's hard zero shows up, and a mean over ten patients that
    # contains one of those is not a mean anyone should quote unqualified.
    seed = next(iter(per))
    rows_pp = sorted(per[seed].items(), key=lambda kv: kv[1])
    print(f"{'pacient':10s} {'Dice 3D':>9s}")
    print("-" * 20)
    for case, value in rows_pp:
        print(f"{case:10s} {value:9.3f}")
    zeros = [c for c, v in rows_pp if v < 1e-9]
    mean_all = sum(per[seed].values()) / len(per[seed])
    print(f"\n  media {mean_all:.4f}")
    if zeros:
        rest = [v for c, v in rows_pp if c not in zeros]
        print(f"  {len(zeros)} pacient(i) la exact 0.000: {', '.join(zeros)}")
        print(f"  fara ei, media ar fi {sum(rest) / len(rest):.4f} "
              f"({sum(rest) / len(rest) - mean_all:+.4f})")
        print(f"  Un zero absolut nu e zgomot - vezi analiza pe lung_036: "
              f"geometrie\n  curata, plafon 0.9467, tumoare vizibila, si modelul "
              f"da 1.7e-06 in\n  masca in timp ce aprinde 0.99 la 100 de felii "
              f"distanta.")
elif len(reference) > 1:
    cases = sorted(next(iter(per.values())))
    seeds = sorted(per)
    head = f"{'pacient':10s}" + "".join(f"{'seed ' + str(s):>10s}" for s in seeds)
    print(head + f"{'ab.std':>10s}")
    print("-" * len(head + " " * 10))
    spreads = {}
    for c in cases:
        vals = [per[s][c] for s in seeds]
        spreads[c] = st.stdev(vals) if len(vals) > 1 else 0.0
        print(f"{c:10s}" + "".join(f"{v:10.3f}" for v in vals)
              + f"{spreads[c]:10.3f}")
    worst = sorted(spreads, key=spreads.get, reverse=True)[:3]
    print(f"\n  Cei mai instabili trei pacienti: "
          + ", ".join(f"{c} ({spreads[c]:.3f})" for c in worst))
    total = sum(spreads.values())
    print(f"  Ei aduna {100 * sum(spreads[c] for c in worst) / total:.0f}% "
          f"din imprastierea totala pe pacienti.")

### Bugetul: sunt 49 390 de pasi lungimea potrivita?

Cifra vine din 14 368 / 16 x 55, unde 55 e numarul de epoci pe care
`baseline/seed_42` s-a intamplat sa le parcurga inainte sa i se termine
rabdarea. Fratii lui au parcurs 47 si 56. Deci variabila de control de sub
fiecare comparatie din proiect e accidentul unei rulari.

Rularea lunga are dublul bugetului, cu cosinusul intins peste el. Nu e aceeasi
rulare continuata: cosinusul se recalca pe buget, deci un buget dublu inseamna
o programare de invatare de doua ori mai lenta, nu aceeasi programare prelungita.

Ce se citeste: **unde cade cea mai buna epoca**. Dincolo de 90 din 110, si tot
ce am masurat pana acum e sub-antrenat. In jur de 50-70, cu o coada plata dupa,
si 49 390 e o conventie pe care o putem apara.

In [ ]:
import csv

CAL = "budget_calibration"
if CAL in runs:
    path = os.path.join(config.OUTPUT_DIR, "experiments", CAL, "seed_42",
                        "training_history.csv")
    rows_h = list(csv.reader(open(path)))
    header, body = rows_h[0], rows_h[1:]
    col = {name: i for i, name in enumerate(header)}
    val = [float(r[col["val_dice_soft"]]) for r in body]
    steps = [int(r[col["optimizer_steps"]]) for r in body]
    best_i = max(range(len(val)), key=lambda i: val[i])

    print(f"  {len(val)} epoci, {steps[-1]:,} pasi")
    print(f"  cea mai buna epoca: {best_i + 1} din {len(val)} "
          f"({100 * (best_i + 1) / len(val):.0f}% din rulare), "
          f"la {steps[best_i]:,} pasi")

    # Where was it when it had spent the reference budget?
    at_ref = min(range(len(steps)), key=lambda i: abs(steps[i] - BASELINE_STEPS))
    before = [v for v, s in zip(val, steps) if s <= BASELINE_STEPS]
    after = [v for v, s in zip(val, steps) if s > BASELINE_STEPS]
    print(f"\n  La {BASELINE_STEPS:,} pasi (epoca {at_ref + 1}): "
          f"val {val[at_ref]:.4f}")
    if before and after:
        print(f"  media val inainte de acel punct : {sum(before)/len(before):.4f}")
        print(f"  media val dupa                  : {sum(after)/len(after):.4f}")

    frac = (best_i + 1) / len(val)
    if frac > 0.85:
        print(f"\n  Cea mai buna epoca e in ultima sesime. Nici bugetul dublu nu")
        print(f"  ajunge - tot ce am masurat pana acum e sub-antrenat, si")
        print(f"  clasamentele dintre configuratii pot fi ale vitezei de")
        print(f"  convergenta, nu ale calitatii finale.")
    elif steps[best_i] <= BASELINE_STEPS:
        print(f"\n  Optimul cade INAINTE de bugetul de referinta. 49 390 de pasi")
        print(f"  sunt suficienti, si probabil generosi.")
    else:
        print(f"\n  Optimul cade dupa bugetul de referinta, la "
              f"{steps[best_i]:,} pasi.")
        print(f"  49 390 taie inainte de optim: comparatiile facute la bugetul")
        print(f"  acela sunt intre modele sub-antrenate, toate in acelasi fel.")
else:
    print("  Rularea de calibrare nu s-a facut - bugetul ramane conventia")
    print("  mostenita de la baseline-ul comis, si trebuie declarat ca atare.")

### Cat din numarul raportat e noroc de selectie

Jitterul validarii scade de vreo douasprezece ori pe masura ce cosinusul se
receste - 0.056 pe epoca la rata mare, 0.005 la eta_min. Un argmax peste o curba
asa cade aproape sigur in jumatatea zgomotoasa, indiferent care model e mai bun,
si toate cele trei seed-uri comise asa au cazut: epocile 35, 27 si 36, niciodata
din coada recoapta.

Greutatile finale sunt deja pe disc. Scorate langa cele alese, dau o marime
acelei distorsiuni, fara nicio antrenare in plus. Pragul se re-baleiaza pe
validare pentru fiecare set de greutati, fiindcă un prag ales pentru unele nu e
pragul potrivit pentru altele.

In [ ]:
from src.evaluation.score_checkpoint import compare_best_and_final, print_comparison

curse = {}
for name in runs:
    print(f"\n{'#' * 70}\n#  {name}\n{'#' * 70}", flush=True)
    try:
        curse[name] = compare_best_and_final(
            f"{name}/seed_{runs[name]['config']['seed']}",
            output_dir=config.OUTPUT_DIR)
        print_comparison(curse[name])
    except Exception:
        traceback.print_exc()

if curse:
    deltas = [c["delta"] for c in curse.values()]
    print(f"\n  Peste {len(deltas)} rulari, final - best: "
          + " ".join(f"{d:+.4f}" for d in deltas))
    if all(d > 0 for d in deltas):
        print("  Toate favorizeaza greutatile recoapte. Selectia pe argmax")
        print("  costa, nu castiga, si fiecare cifra comisa e optimista.")
    elif all(d < 0 for d in deltas):
        print("  Toate favorizeaza epoca aleasa pe validare. Selectia rezista.")

In [ ]:
archive()
print(f"{OUT}.zip  ({os.path.getsize(OUT + '.zip') / 1e6:.0f} MB)")
print(f"Sesiune: {(time.time() - SESSION_START) / 3600:.1f} h")
if failures:
    print("\nEsecuri:", ", ".join(str(k) for k in failures))